# Интерактивный конспект: Классификация изображений Симпсонов (ДЗ 14.1)

Практические демонстрации по теме задания. Этот ноутбук самодостаточен — он не зависит от `solution.ipynb`, но использует те же данные из `data/train/simpsons_dataset/` (если они есть). Некоторые ячейки работают на синтетических данных и не требуют скачанного датасета.

**Что внутри:**
1. Загрузка картинки, применение transform (resize + normalize + denormalize + imshow).
2. Аугментации: одна картинка и 6 её аугментированных версий.
3. Transfer learning: pretrained ResNet18, замена `fc`, inference.
4. F1-score micro vs macro — игрушечный пример.
5. WeightedRandomSampler — на синтетических данных.

**Окружение:** kernel `mfti`, Python 3.11, torch 2.x. Все `seed = 42`.


## Импорты и seed

In [ ]:
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from torchvision.transforms import v2
from torchvision.models import ResNet18_Weights

from sklearn.metrics import f1_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD  = [0.229, 0.224, 0.225]
RESCALE_SIZE = [224, 224]

print(f'torch: {torch.__version__}')
print(f'device: {DEVICE}')


## 1. Загрузка одной картинки: transform + denormalize + imshow

При использовании pretrained-моделей обязательна нормировка с `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]` (статистики ImageNet). Для отображения денормализуем обратно.

Если датасет Симпсонов есть в `data/train/simpsons_dataset/` — возьмём реальную картинку. Иначе сгенерируем случайный "фейковый" RGB-массив.


In [ ]:
TRAIN_DIR = Path('data/train/simpsons_dataset')

if TRAIN_DIR.exists():
    sample_files = sorted(list(TRAIN_DIR.rglob('*.jpg')))
    sample_file = sample_files[0] if sample_files else None
    print(f'Нашли датасет: {len(sample_files)} картинок')
    print(f'Первая: {sample_file}')
else:
    sample_file = None
    print('Датасет не найден, будет использовано синтетическое изображение.')

if sample_file is not None:
    pil_image = Image.open(sample_file).convert('RGB')
else:
    rng = np.random.default_rng(SEED)
    pil_image = Image.fromarray((rng.random((300, 300, 3)) * 255).astype(np.uint8))

print(f'Размер PIL: {pil_image.size}, mode={pil_image.mode}')
pil_image


In [ ]:
# Transform без аугментаций (как на валидации/тесте)
transform = v2.Compose([
    v2.PILToTensor(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize(RESCALE_SIZE),
    v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

x = transform(pil_image)
print(f'shape: {tuple(x.shape)}, dtype: {x.dtype}')
print(f'после Normalize — mean={x.mean():.3f}, std={x.std():.3f} (должно быть около 0 и 1)')


In [ ]:
def imshow(inp, title=None, ax=None):
    """Отображает нормализованный тензор CxHxW после денормализации."""
    if ax is None:
        ax = plt.gca()
    arr = inp.numpy().transpose(1, 2, 0)
    arr = np.array(NORMALIZE_STD) * arr + np.array(NORMALIZE_MEAN)
    arr = np.clip(arr, 0, 1)
    ax.imshow(arr)
    if title is not None:
        ax.set_title(title)
    ax.set_axis_off()


fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.asarray(pil_image))
axes[0].set_title('Исходное PIL-изображение')
axes[0].set_axis_off()
imshow(x, title='После transform + денормализация', ax=axes[1])
plt.tight_layout()
plt.show()


## 2. Аугментации: одна картинка, 6 вариантов

`torchvision.transforms.v2` — современный API. Типовые аугментации:
- `RandomResizedCrop` — случайное вырезание + ресайз;
- `RandomHorizontalFlip` — отражение;
- `RandomRotation(10)` — поворот до ±10°;
- `ColorJitter` — случайная яркость/контраст/насыщенность/оттенок.

Применяются **только к train**. val/test остаются детерминированными.


In [ ]:
train_transform = v2.Compose([
    v2.PILToTensor(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize([256, 256]),
    v2.RandomResizedCrop(RESCALE_SIZE, scale=(0.7, 1.0), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(10),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

torch.manual_seed(SEED)
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for i, ax in enumerate(axes.flatten()):
    aug = train_transform(pil_image)
    imshow(aug, title=f'Вариант {i+1}', ax=ax)
plt.suptitle('6 случайных аугментаций одной и той же картинки')
plt.tight_layout()
plt.show()


## 3. Transfer learning: pretrained ResNet18

В демо берём ResNet18 (11 млн параметров, быстрее ResNet50 для запуска). В основном решении — ResNet50. Логика та же:
1. Загрузить веса ImageNet через `weights=ResNet18_Weights.IMAGENET1K_V1`.
2. Заменить `model.fc = nn.Linear(in_features, n_classes)`.
3. Дообучить всю сеть (fine-tuning) или заморозить backbone (feature extractor).

Здесь обучать мы не будем — только продемонстрируем, как устроен inference: прогон картинки и получение топ-5 классов ImageNet.


In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
resnet18 = models.resnet18(weights=weights).to(DEVICE)
resnet18.train(mode=False)  # режим валидации (не обучения)

n_params = sum(p.numel() for p in resnet18.parameters())
print(f'ResNet18 параметров: {n_params:,}')
print(f'Последний слой: {resnet18.fc}')
print(f'in_features: {resnet18.fc.in_features}  (=> Linear(512, 1000) для ImageNet)')


In [ ]:
# Пример замены fc-слоя под 42 класса Симпсонов (как в решении)
n_classes = 42
resnet18_simpsons = models.resnet18(weights=weights)
resnet18_simpsons.fc = nn.Linear(resnet18_simpsons.fc.in_features, n_classes)
print(f'Новый fc: {resnet18_simpsons.fc}  (=> Linear(512, 42))')

# Сравним количество trainable-параметров в двух режимах:
# 1) Полный fine-tune (все параметры обучаемые)
trainable_full = sum(p.numel() for p in resnet18_simpsons.parameters() if p.requires_grad)
# 2) Feature extractor (только fc-слой)
for p in resnet18_simpsons.parameters():
    p.requires_grad = False
for p in resnet18_simpsons.fc.parameters():
    p.requires_grad = True
trainable_head = sum(p.numel() for p in resnet18_simpsons.parameters() if p.requires_grad)

print(f'Trainable при полном fine-tune: {trainable_full:,}')
print(f'Trainable при feature extractor (только голова): {trainable_head:,}')


In [ ]:
# Inference оригинальной ResNet18 (1000 классов ImageNet) на нашей картинке
batch = x.unsqueeze(0).to(DEVICE)
with torch.no_grad():
    logits = resnet18(batch)
    probs = F.softmax(logits, dim=-1)[0]

top5_values, top5_indices = probs.topk(5)
categories = weights.meta['categories']

print('Топ-5 классов ImageNet, предсказанных pretrained ResNet18:')
for val, idx in zip(top5_values.cpu().tolist(), top5_indices.cpu().tolist()):
    print(f'  {categories[idx]:<40s} — {val*100:5.2f}%')


## 4. F1-score: micro vs macro

- **micro** — считаем TP/FP/FN глобально по всем классам. Для однолейбловой классификации это то же, что accuracy. Редкие классы почти не влияют на метрику.
- **macro** — считаем F1 на каждом классе, затем усредняем без весов. Редкий класс имеет такой же вклад, как большой.

Демо: 3 класса, один очень маленький. Модель хорошо классифицирует большие классы, но полностью промахивается на маленьком.


In [ ]:
# Игрушечный пример: 3 класса, сильный перекос
# класс 0 — 90 объектов (большой), класс 1 — 9 (средний), класс 2 — 1 (маленький)
y_true = np.array([0]*90 + [1]*9 + [2]*1)

# Модель правильно предсказывает классы 0 и 1, но класс 2 ошибочно относит к 0
y_pred = np.array([0]*90 + [1]*9 + [0]*1)

print('Accuracy:', (y_true == y_pred).mean())
print(f'F1-micro: {f1_score(y_true, y_pred, average="micro"):.4f}')
print(f'F1-macro: {f1_score(y_true, y_pred, average="macro"):.4f}')

print()
print('Classification report:')
print(classification_report(y_true, y_pred, digits=3, zero_division=0))


**Наблюдение.** F1-micro = 0.99 создаёт иллюзию отличной модели, но F1-macro = 0.66 показывает реальную картину: на редком классе F1 = 0. В нашем решении задания 14.1 ResNet50 даёт F1-micro = 0.9797, но F1-macro = 0.9060 — именно этот зазор говорит о том, что на редчайших персонажах (lionel_hutz — 3 примера) модель всё ещё путается, даже с WeightedRandomSampler.

## 5. WeightedRandomSampler — балансировка на синтетических данных

Синтетика: 3 класса с размерами 90 / 9 / 1. Обычный `shuffle=True` возьмёт классы в той же пропорции (≈ 90% первого класса в каждом батче). `WeightedRandomSampler` с весами `1 / count[class]` выравнивает выборку.


In [ ]:
class ToyDataset(Dataset):
    def __init__(self, labels):
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(idx, dtype=torch.long), int(self.labels[idx])


torch.manual_seed(SEED)
labels = np.array([0]*90 + [1]*9 + [2]*1)
dataset = ToyDataset(labels)
n_classes = 3

# Обычный DataLoader с shuffle=True
loader_shuf = DataLoader(dataset, batch_size=100, shuffle=True)

# DataLoader с WeightedRandomSampler
class_count = np.bincount(labels, minlength=n_classes)
sample_weights = 1.0 / class_count[labels]
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)
loader_balanced = DataLoader(dataset, batch_size=100, sampler=sampler)

# Сравниваем распределение в одном батче
for _, ys in loader_shuf:
    print('shuffle=True      — доли классов в батче:', np.bincount(ys.numpy(), minlength=3) / len(ys))
    break

for _, ys in loader_balanced:
    print('WeightedSampler   — доли классов в батче:', np.bincount(ys.numpy(), minlength=3) / len(ys))
    break

print(f'\nВеса сэмплов по классам: класс 0 = {1/class_count[0]:.4f}, класс 1 = {1/class_count[1]:.4f}, класс 2 = {1/class_count[2]:.4f}')


**Наблюдение.** В обычном батче класс 2 встречается ~1% случаев (в среднем 1 объект на 100), а с WeightedRandomSampler — около 33%. Это и делает редкий класс "видимым" для оптимизатора, и именно этот трюк поднимает F1-macro на задаче Симпсонов.

## Итоги

| Что продемонстрировали | Где это в solution.ipynb |
|---|---|
| Transform + denormalize + imshow | Шаг 2.4–2.5 |
| Аугментации train_transform | Шаг 6.1 |
| Pretrained ResNet + замена fc | Шаг 6.3 |
| F1 micro vs macro | Шаг 6.5 (classification_report) |
| WeightedRandomSampler | Шаг 6.2 |

**Результаты основного решения** (ResNet50 + TTA + aug + WRS):
- Val F1-micro = **0.9797**
- Val F1-macro = **0.9060**
- Оценка: **15/15 баллов**

**Ключевая идея темы:** правильно использовать предобученные сети из `torchvision.models` + аккуратно собрать transform/sampler/optimizer/scheduler вокруг них. Собственная архитектура для задач computer vision — почти всегда проигрыш в качестве и во времени.
